In [ ]:
context = {
    "input": {
        # 1. 核心数据路径 (请确保这些文件在你的工作目录下)
        "paths": {
            "mlp_tif":"MLP_Scores_5Bands.tif",
            "trans_tif":"Trans_Scores_5Bands.tif",
              # 11波段打分结果 (Band 1是ID)
            "base_tif": "func_shuangliu.tif",       # 现状底图 (用于提取有效掩膜)
            "cz_tif": "cz_shuangliu.tif",                   # 城镇开发边界
            "st_tif": "st_shuangliu.tif",                   # 生态保护红线
            "yj_tif": "yj_shuangliu.tif"                    # 永久基本农田
        },

        #波段名称映射字典
        "MLP_band_names": [
            "MLP_urban_production","MLP_urban_life"
            ,"MLP_ecological_function","MLP_country_production"
            ,"MLP_country_life"
        ],
        "Trans_band_names": [
            "Trans_urban_production","Trans_urban_life"
            ,"Trans_ecological_function","Trans_country_production"
            ,"Trans_country_life"
        ],
    },

    # 留给 AI 存放大表和仿射变换矩阵的空地
    "data": {},

    # 留给 AI 存放最终分配 JSON 的空地
    "output": {}
}
targets = [85474, 273999, 150532, 594191, 77684]

In [ ]:
system_prompt='''
1.角色:
你是一个用于国土空间功能布局优化后端的代码生成助手.
2.任务:
根据内存中全局字典 `context`信息,调用工具 `execute_memory_python_code` 在内存中运行代码,实现指定功能.
3.通用要求:
(1)所有路径与参数必须从 `context['input']` 中读取,绝对不要硬编码.
(2)你的代码不需要被封装在函数里,直接写执行逻辑即可.
(3)必须通过本地修改in-place将结果存入 `context['data']` 或 `context['output']`.
(4)遇到任何报错,请分析 Traceback,修改你的代码并重新调用工具.
'''

In [ ]:
data_description_user = f"""
本项目所用数据均为坐标,范围,分辨率一致的栅格数据,主要包括以下 4 类:
1.MLP 评分栅格(物理适宜度)
(1)含义:表示各像元在物理条件方面对不同国土空间功能的适宜度.
(2)数值范围:0–1，值越大适宜度越高.
(3)数据结构:5 波段栅格数据,各波段含义依次为:
第 1 波段:城镇生产功能物理适宜度;
第 2 波段:城镇生活功能物理适宜度;
第 3 波段:生态功能物理适宜度;
第 4 波段:乡村生产功能物理适宜度;
第 5 波段:乡村生活功能物理适宜度.
2.Trans 评分栅格(空间扩展适宜度)
(1)含义:表示各像元在空间布局与连通条件方面,对不同国土空间功能的扩展适宜度.
(2)数值范围:0–1,值越大适宜度越高
(3)数据结构:5 波段栅格数据,各波段含义依次为:
第 1 波段:城镇生产功能空间扩展适宜度;
第 2 波段:城镇生活功能空间扩展适宜度;
第 3 波段:生态功能空间扩展适宜度;
第 4 波段:乡村生产功能空间扩展适宜度;
第 5 波段:乡村生活功能空间扩展适宜度.
3.空间约束栅格
(1)含义:表示各类重要管控范围内外的空间约束信息.
(2)数值编码:1表示该像元位于相应约束范围内;0表示该像元不在相应约束范围内.
(3)数据结构:3 波段栅格数据,各波段含义依次为:
第 1 波段,城镇开发边界:{context['input']['paths']['cz_tif']}
第 2 波段,生态保护红线:{context['input']['paths']['st_tif']}
第 3 波段,永久基本农田:{context['input']['paths']['yj_tif']}
4.现状功能类型栅格
(1)含义:表示各像元当前的国土空间功能类型.
(2)数据结构:为单波段栅格数据,像元值为功能类型编码,含义为:
1:城镇生产功能;
2:城镇生活功能;
3:生态功能;
4:乡村生产功能;
5:乡村生活功能.
"""

In [ ]:
data_description_sys = f"""
掩膜提取栅格:{context['input']['paths']['base_tif']}
 - 总波段数:1
 - 提取唯一波段作为“现状功能类型”的一维数组
 - 无效背景值:-9999 或 <= 0 或 15
 - 有效功能类型数值: 1,2,3,4,5
"""

In [ ]:
data_description = data_description_user + data_description_sys

In [ ]:
step1_prompt = f"""
1.你是一名编程助手,请根据规划数据说明[{data_description}],编写python代码,完成以下任务:
(1)使用 rasterio 或等效库读取{context['input']}中的所有栅格数据,以{context['input']['paths']['mlp_tif']}栅格作为基准,检查所有栅格:
 - 行列数是否一致;
 - 仿射变换是否一致;
 - 坐标参考系统是否一致;
 - 若不一致，抛出异常.
(2)数据清洗:
在进行掩膜过滤和数据清洗前,必须先将所有读取到的栅格矩阵在空间维度上展平,即:
 - 单波段栅格展平为一维数组;
 - 多波段栅格展平为二维数组;
 - 禁止在 3D 状态下应用布尔掩膜;
 - 必须建立绝对有效的栅格像元;
 - 过滤掉现状功能类型为无效值(<=0 或 ==-9999)的行,以及任何 像元值为-9999或NaN的行.
(3)对现状功能类型进行统计,得到每个功能类型对应的像元数,保存到context["status_counts"]中.
2.输出要求:
(1)context["raster_meta"]:包含高度,宽度,仿射变换基础元数据(这些数据保存到context中);
(2)`context`仅用于存储轻量级变量,严禁将大型数据实体存入context;
(3)结果保存在step1_base_table.csv本地文件中,并将该文件存储目录保存在context['output_files']['base_table'] = 'step1_base_table.csv';
(4)对step1_base_table.csv文件,在完成数据清洗后,重新生成首列 id(该列必须为从 0 到 n-1的连续整数(int)序列,其中n 为清洗后剩余的有效像元总数).
3.输出完整可执行的 Python 代码(该代码不要附加解释性文字),并调用 `execute_memory_python_code` 工具在内存中执行. 
"""
print(f"prompt:{step1_prompt}")

In [ ]:
import io
import traceback
import contextlib

# =====================================================================
# 🔧 核心：内存级 Python 执行工具 (操作 Context)
# =====================================================================
def execute_memory_python_code(code: str) -> str:
    """在当前内存中执行代码，直接修改全局的 context，并拦截 print 输出或报错"""
    print("\n  [🔧 触发 Tool：在内存中热执行 AI 代码...]")

    # 准备一个内存缓冲区，用来拦截 print 的内容
    f = io.StringIO()

    try:
        # 重定向标准输出，将 print 内容拦截到 f 中
        with contextlib.redirect_stdout(f):
            # 💡 核心：定义执行环境，把本地的 context 和 GIS 常用库直接“喂”给沙盒
            exec_globals = {
                "context": context,
                "pd": __import__('pandas'),
                "np": __import__('numpy'),
                "rasterio": __import__('rasterio')
            }
            # 在当前内存中执行代码
            exec(code, exec_globals)

        output = f.getvalue()
        if output.strip():
            # 截断过长的输出，防止把大模型的上下文撑爆
            if len(output) > 2000:
                output = output[:2000] + "\n...[输出过长已截断]..."
            return f"✅ 代码执行成功！\n终端输出如下:\n{output}"
        else:
            return "✅ 代码执行成功！(无 print 输出)"

    except Exception as e:
        # 抓取最底层的报错堆栈，原封不动还给 AI 查错
        error_trace = traceback.format_exc()
        return f"❌ 执行报错！请分析报错并修改代码重试:\n{error_trace}"

# =====================================================================
# 🛠️ 定义 Tool Schema (发给大模型的“工具说明书”)
# =====================================================================
tools = [{
    "type": "function",
    "function": {
        "name": "execute_memory_python_code",
        "description": "在内存中执行 Python 代码。环境已预置了全局变量 `context` (字典)，以及 `pd`, `np`, `rasterio` 库。你可以直接读取并修改 `context`。",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "要执行的完整 Python 代码。直接编写执行逻辑，无需封装在函数内。"
                }
            },
            "required": ["code"]
        }
    }
}]


In [ ]:
import  os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
print(find_dotenv())
print(os.getcwd())
print(os.getenv("OFOX_API_KEY"))
print(repr(os.getenv("OFOX_API_KEY")))



In [ ]:
import os
import io
import httpx
import traceback
import contextlib
import anthropic
from dotenv import load_dotenv, find_dotenv

# =====================================================================
# 1. 初始化客户端与全局 Context (已切换为 Claude)
# =====================================================================
load_dotenv(find_dotenv(), override=True)

# 初始化 Anthropic 客户端
client = anthropic.Anthropic(
    api_key=os.getenv("OFOX_API_KEY"),
    base_url="https://api.ofox.io/anthropic",
    timeout=1800.0
)

# 确保 context 中有一个专门用于存放 Agent 记忆的区域
if 'agent_memory' not in context:
    context['agent_memory'] = {}

def execute_memory_python_code(code: str) -> str:
    print("\n  [🔧 触发 Tool：在内存中热执行 AI 代码...]")
    f = io.StringIO()
    try:
        with contextlib.redirect_stdout(f):
            # 预埋好环境变量和常用库
            exec_globals = {
                "context": context,
                "pd": __import__('pandas'),
                "np": __import__('numpy'),
                "rasterio": __import__('rasterio')
            }
            exec(code, exec_globals)

        output = f.getvalue()
        if output.strip():
            if len(output) > 2000:
                output = output[:2000] + "\n...[输出过长已截断]..."
            return f"代码执行成功！\n终端输出如下:\n{output}"
        else:
            return "代码执行成功！(无 print 输出)"

    except Exception as e:
        error_trace = traceback.format_exc()
        return f"执行报错！请分析报错并修改代码重试:\n{error_trace}"

#  Claude 专属的 Tool 结构定义
tools = [{
    "name": "execute_memory_python_code",
    "description": "在内存中执行 Python 代码。环境已预置全局变量 `context` 及 `pd`, `np`, `rasterio` 库。可直接修改 `context`。",
    "input_schema": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "要执行的 Python 代码。无需封装函数。"
            }
        },
        "required": ["code"]
    }
}]

print("启动 Step 1 空间底表构建 Agent (Claude 引擎驱动)...")

# 注意：Claude 的 messages 列表里不能放 system，只能放 user 和 assistant
messages = [
    {"role": "user", "content": step1_prompt}
]

max_steps = 5  # 给它 5 次改错的机会
final_report = "" # 用于保存最终汇报内容

for step in range(max_steps):
    print(f"\n" + "━"*50)
    print(f"🔄 第 {step+1} 轮：AI 思考与行动")
    print("━"*50)

    # 发送请求给 Claude
    response = client.messages.create(
        model="anthropic/claude-sonnet-4.6", # 你的带思考过程的模型
        system=system_prompt,               # System 提示词作为单独参数传入
        messages=[{"role": "user", "content": step1_prompt}],
        tools=tools,
        max_tokens=4096                     # Claude API 必须提供 max_tokens
    )

    # 将助手的完整回复（包含 text, thinking, tool_use 等所有区块）原封不动加入历史记录
    messages.append({"role": "assistant", "content": response.content})

    # 判断模型是否决定调用工具
    if response.stop_reason == "tool_use":
        tool_results_content = []

        # 遍历回复中的每一个数据块，找到所有的工具调用请求
        for block in response.content:
            if block.type == "thinking":
                print(f"🤔 [Claude 内部思考中]...\n{block.thinking[:200]}...\n") # 打印前200字思考过程

            elif block.type == "tool_use":
                func_args = block.input # Claude SDK 直接返回字典，不需要 json.loads
                ai_code = func_args.get("code", "")

                print(f"💻 [AI 编写的代码]:\n{'-'*30}\n{ai_code}\n{'-'*30}")

                # 执行代码并获取反馈
                tool_result = execute_memory_python_code(ai_code)
                print(f"📊 [执行反馈]:\n{tool_result}")

                # Claude 专属：将工具执行结果包装成 tool_result 块
                tool_results_content.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": tool_result
                })

        # 将本次所有的工具执行结果，作为 user 角色发送给模型，让它继续
        messages.append({"role": "user", "content": tool_results_content})

    else:
        # 如果不是 tool_use，说明模型决定结束任务并输出最终文本了
        final_text = "\n".join([block.text for block in response.content if block.type == "text"])
        print(f"\n🎉 AI 汇报：\n{final_text}")

        # 记录最终汇报
        final_report = final_text
        break


context['agent_memory']['step1'] = {
     "messages": messages          # 如果需要保存整个对话历史，取消注释这行（但注意可能较长）
}

print("\n📦 [运行后] 你的终极 context 状态：", context)

In [ ]:
context

## 第二步

In [ ]:
planning_requirements = f"""
1.规划目标面积(规划像元个数):
城镇生产(如未填,则目标面积 = 现状面积):{targets[0]} 个
城镇生活(如未填,则目标面积 = 现状面积):{targets[1]} 个
生态功能(如未填,则目标面积 = 现状面积):{targets[2]} 个
乡村生产(如未填,则目标面积 = 现状面积):{targets[3]} 个
乡村生活(如未填,则目标面积 = 现状面积):{targets[4]} 个
2.规划原则
1)现状保护原则
若像元现状功能 = 规划目标功能,在同等条件下优先选择保持现状,减少不必要转换.
2)保护区避让原则
(1) 对城镇生产,城镇生活,乡村生活功能:
优先使用不在永久基本农田,不在生态红线内的像元;可选像元不足时,允许从永久基本农田,生态红线范围内补充.
(2)对乡村生产功能:
优先使用不在生态红线,不在城镇开发边界内的像元;可选像元不足时,允许从生态红线,城镇开发边界内补充.
(3) 对生态功能:
优先使用不在永久基本农田,不在城镇开发边界内的像元;可选像元不足时,允许从永久基本农田,城镇开发边界内补充.
3)同等条件协调原则
保护区重要程度排序(从高到低):永久基本农田保护＞生态保护＞城镇开发边界.
适宜度重要程度排序(从高到低):物理适宜度(MLP评分)＞空间扩展适宜度(Trans评分).
3.规划模型构建要求
(1) 优先采用线性规划模型。
(2) 以实现综合适宜度得分总和最大化为目标。
(3) 综合适宜度得分计算公式为：
    - 综合适宜度(Score) = 权重1(W_mlp)*MLP评分(U_mlp) + 权重2(W_trans)*Trans评分(U_trans) - 权重3(W_avoid)*违背避让原则扣分(U_avoid) - 权重4(W_redline)*违背协调原则扣分(U_redline) + 权重5(W_status)*遵守现状保护原则加分(U_status)
    - 权重和为1，即：W_mlp+W_trans+W_avoid+W_redline+W_status=1.
    - 权重1与权重2分别是MLP适宜度评分和trans适宜度评分，二者数值相当，权重5作为现状保护得分，应获得最高的权重值，权重3作为违背避让原则的扣分权重应高于权重4作为违背协调原则的扣分权重，比如：权重1(0.15),权重2(0.15),权重3(0.25)，权重4(0.2)，权重5(0.25)
(4)加分、扣分规则
    - 分值范围为0-1，其中0-0.3表示低值，0.4-0.6是中值，0.7-1.0表示高值；
    - U_mlp，U_trans分别通过 `np.argsort` 两次排序，将其映射为 [0, 1] 的全局竞争力得分；
    - 遵守现状保护原则加分规则：城镇生产，城镇生活赋高值，生态赋中值或高值，但要低于城镇得分，乡村生产和生活赋低值；
    - 违背协调原则扣分规则：按照协调优先级顺序扣分，比如：违背保护区重要程度排序（从高到低），永久基本农田保护、生态保护、城镇开发边界依次扣分为 0.7、0.5、0.3；
    - 违背避让原则扣分规则：违背任何避让原则都扣相同分值为，比如：1。
"""
print(planning_requirements)

In [ ]:
step2_prompt = f"""
1.你是一名国土空间规划规则分析助手,请根据规划要求[{planning_requirements}]，结合上下文信息[{context["agent_memory"]["step1"]}],分析并输出一个结构化的“模型配置对象”(JSON 格式或 Python 字典文本),字段包括但不限于:
(1)targets:对应各类功能的“规划像元个数(整数)”,若用户未给出明确数量,则使用{context["status_counts"]}作为规划像元数量.
(2)planning_rules:用于描述在“规划原则”要求下各功能在不同情景下的奖励、惩罚策略.
(3)model_type:用户指定的规划模型类型(如"MILP","GA" 等),若用户未指定模型,由你根据数据规模与规划要求筛选确定最佳的模型.
(4)Data_Analysis_Results:分析用户数据得到的优化建模需要的重要信息,如各类功能适宜度权重等.
(5)其他你通过分析规划要求和上下文信息认为对优化建模重要的信息,可自行建立字段名并存储相关数据.
(6)若用户指定了在当前数据和能力下无法实现的模型或规则,请在context中加入字段 error,说明原因.
2.输出要求：
(1)不要写具体实现代码,只给出数据结构,为后续空间优化模型构建和编写代码提供支撑。
(2)输出必须是一个完整的JSON文本，便于大语言模型解析与理解。
(3)禁止输出各种注释,如://或#等。
"""
print(f"stage2 prompt:{step2_prompt}")

In [ ]:
import os
import re
import json
import ast  # 👈 核心：引入 Python 的抽象语法树模块，用于智能解析
import httpx
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

print("🚀 启动 Step 2：规划规则解析 Agent (切换至 gpt-5.1 引擎)...")

load_dotenv(find_dotenv(), override=True)

api_key = os.getenv("OFOX_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.ofox.io/v1",
    http_client=httpx.Client(timeout=1800.0)
)

# =====================================================================
# 🚨 核心修复：重新定义 OpenAI 格式的 Tools，覆盖掉 Claude 的格式！
# =====================================================================
tools = [{
    "type": "function",
    "function": {
        "name": "execute_memory_python_code",
        "description": "在内存中执行 Python 代码。环境已预置全局变量 `context` 及 `pd`, `np`, `rasterio` 库。可直接修改 `context`。",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "要执行的 Python 代码。无需封装函数。"
                }
            },
            "required": ["code"]
        }
    }
}]

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": step2_prompt}
]

max_steps = 3
for step in range(max_steps):
    print(f"\n" + "━"*50)
    print(f"🔄 第 {step+1} 轮：AI 思考与行动")
    print("━"*50)

    # 发送请求给 gpt-5.1
    response = client.chat.completions.create(
        model="openai/gpt-5",
        messages=messages,
        tools=tools,         # 👈 现在这里用的是正宗的 OpenAI 格式 tools
        tool_choice="auto",
        temperature=0.1
    )

    assistant_msg = response.choices[0].message
    messages.append(assistant_msg)

    # 处理 Tool 调用（OpenAI 格式）
    if assistant_msg.tool_calls:
        for tool_call in assistant_msg.tool_calls:
            try:
                func_args = json.loads(tool_call.function.arguments)
            except Exception as e:
                fixed_str = tool_call.function.arguments.replace('\n', '\\n').replace('\r', '\\r').replace('\t', '\\t')
                try:
                    func_args = json.loads(fixed_str)
                except:
                    messages.append({"role": "tool", "tool_call_id": tool_call.id, "name": tool_call.function.name, "content": "参数解析错误。"})
                    continue

            ai_code = func_args.get("code", "")
            print(f"💻 [AI 编写的代码]:\n{'-'*30}\n{ai_code}\n{'-'*30}")
            tool_result = execute_memory_python_code(ai_code)
            print(f"📊 [执行反馈]:\n{tool_result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": tool_result
            })
    else:
        # 提取最终文本
        print(f"\n🎉 AI 汇报 (原始文本)：\n{assistant_msg.content}")

        # =====================================================================
        # 5. 🚨 终极防线：智能提取并双重容错解析
        # =====================================================================
        content = assistant_msg.content
        match = re.search(r'\{.*\}', content, re.DOTALL)
        if match:
            clean_json_str = match.group(0)

            # 先去除非法的 // 注释，这在标准 JSON 中是不允许的
            clean_json_str = re.sub(r'//.*', '', clean_json_str)

            try:
                # 第一道防线：客客气气地尝试标准 JSON 解析
                parsed_config = json.loads(clean_json_str)
                context['planning_config'] = parsed_config
                print("\n✅ 拦截成功！已通过标准 JSON 提取并注入 context['planning_config']！")

            except json.JSONDecodeError:
                # 第二道防线：如果因为单引号、Python 布尔值等报错，启用 AST 引擎暴力兜底
                print("\n⚠️ 标准 JSON 解析失败，启动 Python AST 智能解析引擎...")
                try:
                    # 将 JSON 的小写关键字转换为 Python 可识别的首字母大写关键字
                    python_dict_str = clean_json_str.replace('true', 'True').replace('false', 'False').replace('null', 'None')
                    parsed_config = ast.literal_eval(python_dict_str)
                    context['planning_config'] = parsed_config
                    print("✅ 拦截成功！已通过 AST 引擎智能解析并注入 context['planning_config']！")
                except Exception as e:
                    print(f"\n❌ 双重解析均告失败: {e}")
                    print("⚠️ 出错的原始字符串是:\n", clean_json_str)
        else:
            print("\n❌ 未能在 AI 回复中找到合法的 JSON 大括号结构。")

        break # 提取完毕，退出循环

print("\n📦 [运行后] 你的终极 context 状态：")
if 'planning_config' in context:
    print("🏆 伟大胜利！context['planning_config'] 成功捕获！其内容为：")
    # 为了好看地打印出来，我们用 json.dumps。如果是 ast 解析的，里面可能有 None，所以要处理一下
    print(json.dumps(context['planning_config'], indent=2, ensure_ascii=False, default=str))
else:
    print("❌ 注入失败，请检查 AI 逻辑。")

In [ ]:
context['planning_config']["model_type"]

In [ ]:
step3_prompt = f"""
1.你是一名数学建模和编程助手,请根据模型配置对象[{context['planning_config']}],结合上下文信息[user:{step2_prompt}],编写python代码,完成以下任务:
1)使用pandas从{context['output_files']['base_table']}指定的CSV文件中提取以下数组:
(1)mlp:二维数组(n_cells, 5),列名是{context['input']['MLP_band_names']};
(2)trans:二维数组(n_cells, 5),列名是{context['input']['Trans_band_names']};
(3)Current_use:一维数组 (n_cells,),读取后务必使用.astype(int)强制转换为整型;
(4)CZ_boundary,ST_redline,YJ_farmland:一维数组 (n_cells,);
(5)model_config:读取context['planning_config']存储的模型相关信息,包括但不限于:targets,planning_rules,Data_Analysis_Results,model_type等.
2.执行工作流程
1)计算各像元各功能综合得分
  根据mlp,trans,current_func和model_config中的权重参数,计算每个像元对每个功能的综合得分;
  根据planning_rules和Data_Analysis_Results信息,辨识哪些像元对于哪些功能是禁止的(设计惩罚机制),对于哪些功能是鼓励的(设计奖励机制).
2)构建优化模型
  (1)根据{context['planning_config']["model_type"]}选择模型类型:
    - 若用户未指定模型类型(model_type为空),则使用合适的线性/整数规划库(优先考虑pulp库),构建决策变量,目标函数和约束条件;
    - 若用户指定了规划模型(例如遗传算法等),则根据模型配置对象和上下文信息设计优化模型与求解算法.
  (2)无论采用哪种建模方法,都必须满足以下硬性约束:
    - 每个像元只能选择一个功能;
    - 每个功能的像元数满足targets要求;
    - 用户提出的确定性的禁止条件.
  (3)优化模型最终目的是确定满足约束条件且使得目标函数最大化的各像元的功能类型.
3)模型求解与结果生成
  (1)执行模型求解，并将求解得到的各像元功能值和像元编号(id)存储到数组new_func;
  (2)检查各功能的像元数是否满足targets以及硬约束,若存在偏差,请重做.
4)输出
  (1)以id编号作为匹配依据，将new_func中存储的各像元的功能类型编号作为一个新列加入到CSV文件的DataFrame中(列名为'new_func'),并将更新后的DataFrame保存为本地文件'step3_result_table.csv';
  (2) 空间元数据获取: 使用 rasterio 读取 context['input']['paths']['base_tif']、'mlp_tif' 和 'trans_tif'，获取 crs、transform、height、width，并严格依据有效值(>0 且 != -9999 且非 nan)重建 valid_mask;
  (3) 2D矩阵还原与 TIF 导出:
   将 id、Current_use 和 new_func 这三个1D数组, 利用 valid_mask 还原为 2D 矩阵 (背景填充为 -9999)。
   导出单波段 TIF 文件 'Final_Planned_Function.tif'，写入 new_func 的 2D 矩阵。
  导出多波段 TIF 文件 'Final_Change_Map_3Bands.tif'，写入三个波段 (Band 1: Cell_ID, Band 2: Current_Type, Band 3: Planned_Type)。
  (4) 更新上下文 Context 参数 (关键交接):
   必须在 context['output_files'] 中记录: 'result_table', 'planned_tif', 'change_tif' 的准确路径。
   必须在 context['data'] 中记录下一步绘图所需的确切列名:
     context['data']['current_func_col'] = '<你提取现状使用的实际列名>'
    context['data']['new_func_col'] = 'new_func'
3.代码编写要求
  (1)必须在当前代码末尾,将下一次需要用到的中间变量(如,score_matrix,best_chrom等)保存到`context['data']`中(如,`context['data']['score_matrix'] = score_matrix`);
  (2)代码中不要调用大语言模型(LLM);
  (3)Python代码编写中不封装函数.
  (4)输出完整可执行的Python代码，并调用 `execute_memory_python_code` 工具在内存中执行，遇到报错(Traceback)请自行修改重运行。
4.退出与返回机制：
  (1) 当你调用 `execute_memory_python_code` 后，观察到【执行成功】、面积约束满足，且 2 个 TIF 文件均成功导出，即代表任务彻底完成！
  (2) 此时，你绝对禁止再次调用任何工具（Tool）或生成新的代码！
  (3) 立即跳出思考循环，用自然语言向用户输出【最终总结报告】，必须包括：
      - 明确宣告 "Step 3 优化求解与空间重构已圆满完成！"
      - 汇报各个功能的最终分配像元数及现状保持率。
      - 确认 CSV 和 2 个 TIF 文件已生成。
      - 明确列出存入 context 的 current_func_col 和 new_func_col 变量值。
  (4) 输出报告后，立刻停止生成，结束本轮对话。
"""
print(f"prompt:{step3_prompt}")

In [ ]:
import os
import json
import anthropic
from dotenv import load_dotenv, find_dotenv

print("🚀 启动 Step 3：PuLP 建模与运筹学求解 Agent (重置为纯净 Claude 引擎)...")

# =====================================================================
# 1. 强制重置客户端为 Claude (Anthropic)
# =====================================================================
load_dotenv(find_dotenv(), override=True)
client = anthropic.Anthropic(
    api_key=os.getenv("OFOX_API_KEY"),
    base_url="https://api.ofox.io/anthropic",  # 你的聚合平台地址
    timeout=1800.0
)
print("✅ 客户端已强行重置为 Anthropic 引擎！")

# =====================================================================
# 2. 强制重置 Tools 为 Claude 专属的 input_schema 格式
# =====================================================================
tools = [{
    "name": "execute_memory_python_code",
    "description": "在内存中执行 Python 代码。环境已预置全局变量 `context` 及 `pd`, `np`, `rasterio` 库。可直接修改 `context`。",
    "input_schema": {  # 👈 Claude 必须用 input_schema，而不是 parameters
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "要执行的 Python 代码。无需封装函数。"
            }
        },
        "required": ["code"]
    }
}]
print("✅ Tools 格式已重置为 Claude 标准！")

# =====================================================================
# 3. 组装 Messages
# =====================================================================
messages = [
    {"role": "user", "content": step3_prompt}
]

# =====================================================================
# 4. 极简循环调用引擎 (给它 6 次试错机会)
# =====================================================================
max_steps = 20
for step in range(max_steps):
    print(f"\n" + "━"*50)
    print(f"🔄 第 {step+1} 轮：AI 思考、建模与求解")
    print("━"*50)

    # 🚨 调用纯正的 Claude 模型
    response = client.messages.create(
        model="anthropic/claude-sonnet-4.6",
        system=system_prompt,               # System prompt 单独传
        messages=messages,
        tools=tools,
        temperature=0.1,                    # 极低温度，保证数学逻辑严谨
        max_tokens=4096                     # 必须有
    )

    # 将助手的完整回复加入历史记录
    messages.append({"role": "assistant", "content": response.content})

    # 判断模型是否决定调用工具写代码
    if response.stop_reason == "tool_use":
        tool_results_content = []

        # 遍历回复中的每一个数据块
        for block in response.content:
            if block.type == "thinking":
                # 打印内心戏
                print(f"🤔 [Claude 内部数学推导中]...\n{block.thinking[:300]}...\n")

            elif block.type == "tool_use":
                func_args = block.input
                ai_code = func_args.get("code", "")

                print(f"💻 [AI 编写的 PuLP 代码]:\n{'-'*30}\n{ai_code}\n{'-'*30}")

                # 🚀 执行代码！(终端可能会卡一会儿，求解器正在运行)
                tool_result = execute_memory_python_code(ai_code)
                print(f"📊 [执行反馈]:\n{tool_result}")

                # 包装结果
                tool_results_content.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": tool_result
                })

        # 将执行结果作为 user 角色发送回给模型
        messages.append({"role": "user", "content": tool_results_content})

    else:
        # 如果 stop_reason 不是 tool_use，说明代码跑通了，或者决定停止了
        final_text = "\n".join([block.text for block in response.content if block.type == "text"])
        print(f"\n🎉 AI 最终汇报：\n{final_text}")
        break

# =====================================================================
# 5. 检查终极战果
# =====================================================================
print("\n📦 [运行后] 检查你的 context 注册表：")
if 'output_files' in context and 'result_table' in context['output_files']:
    print("🏆 伟大胜利！终极分配底表已成功落盘！路径为：", context['output_files']['result_table'])
else:
    print("⚠️ 似乎还没有生成 result_table，请检查上方是否有报错信息。")

if 'agent_memory' not in context:
    context['agent_memory'] = {}

context['agent_memory']['step3'] = {
    # 选项 A：只存 AI 最后的文字总结（推荐！写报告最需要这个）
    "summary_text": final_text,

    # 选项 B：把整个 12 轮的斗智斗勇聊天记录全存下来（以防你需要查看细节）
    "full_messages": messages
}

print("\n✅ Step 3 的 AI 记忆已成功封存至 context['agent_memory']['step3']！")

In [ ]:
context['agent_memory']['step3']['full_messages']

In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

print("🎨 启动 Step 4：基于 Context 成果的极速空间渲染引擎...")

# =====================================================================
# 1. 优雅地从 context 提取核心资产
# =====================================================================
try:
    # 直接提取 Step 3 做好的 3波段 变化追踪图
    change_tif_path = context['output_files']['change_tif']

    # 如果你想在图表标题上打印列名，也可以提取它们
    current_col_name = context['data'].get('current_func_col', '现状功能')
    new_col_name = context['data'].get('new_func_col', '规划功能')

    print(f"📦 成功获取 Context 资产！正在读取空间矩阵: {change_tif_path}")
except KeyError as e:
    raise ValueError(f"❌ 找不到 Context 键值: {e}，请确保 Step 3 已圆满完成并正确保存了 Context！")

# =====================================================================
# 2. 直接读取 2D 矩阵 (跳过复杂的掩膜重建)
# =====================================================================
NODATA = -9999

with rasterio.open(change_tif_path) as src:
    # Band 2: Current_Type (改变前类型)
    current_matrix = src.read(2)
    # Band 3: Planned_Type (改变后类型)
    planned_matrix = src.read(3)

# 识别出有效像元区域 (去除 -9999 的背景边框)
valid_mask = (current_matrix != NODATA)

# 提取真正发生改变的像素区域
changed_mask = (current_matrix != planned_matrix) & valid_mask
actual_changes = int(np.sum(changed_mask))

print(f"📊 空间矩阵加载完毕！共识别到 {actual_changes} 个像元发生了功能变更。")

# =====================================================================
# 3. 准备可视化底图 (背景全置为 0，有效像元填入真实类别 1~5)
# =====================================================================
show_orig = np.zeros_like(current_matrix, dtype=np.uint8)
show_final = np.zeros_like(planned_matrix, dtype=np.uint8)
change_flag_matrix = np.zeros_like(current_matrix, dtype=np.uint8)

# 将有效数据填入画板
show_orig[valid_mask] = current_matrix[valid_mask]
show_final[valid_mask] = planned_matrix[valid_mask]

# 生成变动提取图 (仅变动区为 1，其他为 0)
change_flag_matrix[changed_mask] = 1

# 生成全局变动高亮图 (在现状图的基础上，将变动区强行赋值为 6 以便高亮显示)
show_pred = show_orig.copy()
show_pred[changed_mask] = 6

# =====================================================================
# 4. 核心渲染与四宫格拼图
# =====================================================================
print("🖌️ 正在绘制空间布局四宫格图...")
plt.rcParams['font.sans-serif'] = ['SimHei']  # 正常显示中文
plt.rcParams['axes.unicode_minus'] = False    # 正常显示负号
CLASS_NAMES = {1: "城镇生产", 2: "城镇生活", 3: "生态功能", 4: "乡村生产", 5: "乡村生活"}

# 设置硬核调色盘 (0是白色背景，1~5是类别颜色，6是高亮色)
cmap_main = ListedColormap(['#FFFFFF', '#D7191C', '#FDAE61', '#1A9641', '#FFFFBF', '#2B83BA'])
highlight_color = '#FF00FF'
cmap_highlight = ListedColormap(list(cmap_main.colors[:6]) + [highlight_color])
cmap_changes = ListedColormap(['#FFFFFF', '#00FF00'])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
main_title = f"基于优化算法的空间布局重构\n[成功重分配像元数: {actual_changes} 个]"
fig.suptitle(main_title, fontsize=22, fontweight='bold', y=0.96)

axes[0, 0].set_title(f"1. 现状图 ({current_col_name})", fontsize=16)
axes[0, 0].imshow(show_orig, cmap=cmap_main, vmin=0, vmax=5, interpolation='nearest')
axes[0, 0].axis('off')

axes[0, 1].set_title(f"2. 最终布局图 ({new_col_name})", fontsize=16)
axes[0, 1].imshow(show_final, cmap=cmap_main, vmin=0, vmax=5, interpolation='nearest')
axes[0, 1].axis('off')

axes[1, 0].set_title("3. 全局变动高亮图", fontsize=16)
axes[1, 0].imshow(show_pred, cmap=cmap_highlight, vmin=0, vmax=6, interpolation='nearest')
axes[1, 0].axis('off')

axes[1, 1].set_title("4. 纯净变动提取图", fontsize=16)
axes[1, 1].imshow(change_flag_matrix, cmap=cmap_changes, vmin=0, vmax=1, interpolation='nearest')
axes[1, 1].axis('off')

# 构建底部图例
legend_patches = []
for i, color in enumerate(cmap_main.colors):
    if i == 0: continue # 跳过 0 (背景色)
    ec = 'gray' if i == 4 else 'none'
    legend_patches.append(mpatches.Patch(facecolor=color, edgecolor=ec, label=CLASS_NAMES.get(i, f"类别 {i}")))
legend_patches.append(mpatches.Patch(color=highlight_color, label="功能变动区"))

fig.legend(handles=legend_patches, loc='lower center', ncol=6, fontsize=14, bbox_to_anchor=(0.5, 0.02))
plt.tight_layout(rect=[0, 0.08, 1, 0.95])

# 保存并展示
save_path = "Step4_Final_Optimization_Map.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ 完美出图！图像已保存至本地: {save_path}")

In [ ]:
step4_prompt=f"""
你是一名国土空间规划报告撰写助手,请根据上下文信息[{context['agent_memory']['step3']['full_messages']}],用中文撰写一份“规划说明文字”，用于保存为 Word 文档。
要求：
1.给出必要的背景介绍,明确说明本轮优化采用的方法和模型.
2.用较为详细的步骤，描述本轮大语言模型(LLM)驱动的优化方法的实现过程,包括但不限于:
(1)规划数据与预处理(主要介绍使用了哪些数据,做了哪些预处理等);
(2)功能适宜度评价方法(用数学表达式呈现);
(3)国土功能空间优化布局模型构建(用数学表达式呈现方法原理及步骤);
(4)模型求解的算法流程与具体实现步骤。
3.详细说明各功能类型的规划目标与实际结果的对比情况：
(1)哪些功能目标完全达到，哪些存在偏差;
(3)如存在偏差,请结合模型约束和适宜度分布解释可能原因.
4.概括国土空间功能优化布局的变化特点：
(1)哪些功能的空间分布显著扩张或收缩;
(2)与生态保护,永久基本农田保护,城镇开发边界等约束的匹配情况等.
5.全文应具备正式的技术报告风格,适合作为Word文档直接提交和归档;条理清晰,段落分明;数学公式用标准形式表示,使用常见数学符号,不需要特定排版标记.
6.只输出完整的中文说明文字内容,不要输出任何代码和控制指令.
"""

In [ ]:
import os
import httpx
from openai import OpenAI

print("🚀 启动 Step 5：撰稿 Agent (重新初始化 gpt-5.1 引擎)...")

# 1. 重新定义专门给聚合平台用的客户端 (避免和前面的客户端冲突)
# 🚨 注意：这里沿用你之前测试成功的聚合平台 Key 和带 /v1 的网址
gpt_client = OpenAI(
    api_key=os.getenv("OFOX_API_KEY"),  # 聚合平台通用的 Key
    base_url="https://api.ofox.io/v1",  # OpenAI 兼容格式必须带 /v1
    http_client=httpx.Client(timeout=1800.0) # 防止写长文超时
)

# 2. 去掉系统提示词，直接传入用户的提示词
messages = [
    {"role": "user", "content": step4_prompt}
]

print("⏳ 正在研读前序战报，奋笔疾书中 (大概需要几十秒到几分钟)...")

# 3. 使用新的 gpt_client 调用模型
response = gpt_client.chat.completions.create(
    model="gpt-5.1",  # 👈 明确指定使用 gpt-5.1
    messages=messages,
    temperature=0.3,  # 稍微给一点温度，让文章的语言更丰富自然，但总体保持严谨
)

final_report_text = response.choices[0].message.content

# =====================================================================
# 4. 打印并保存到本地 Markdown 文件
# =====================================================================
print("\n" + "═"*60)
print("🎉 《国土空间优化规划成果说明书》撰写完成！")
print("═"*60 + "\n")
print(final_report_text)

# 自动落盘存为一个 markdown 文件，方便后续转 Word
report_filename = "Final_Planning_Report.md"
with open(report_filename, "w", encoding="utf-8") as f:
    f.write(final_report_text)

print(f"\n✅ 报告已自动保存到本地：{report_filename}")
print("💡 下一步你可以直接运行刚才的 Pandoc 代码，把它一键转换成 Word 文档啦！")

In [ ]:
step5_qwen_prompt = f"""
1.角色:你是一名国土空间规划报告撰写高级助手。请根据以下要求，对提供的初稿内容进行语言润色.
2.任务:参考上轮报告编制信息:[{step4_prompt}],对初稿内容[{final_report_text}]进行专业化润色.
3.角色目标:
(1)语言表达需符合中文专业报告标准：逻辑清晰,层次分明,术语准确,行文严谨;
(2)不得删减任何数学原理和计算公式,必须完整保留所有技术细节;
(3)确保所有数学公式在 Microsoft Word 文档中可正确显示(建议使用 Unicode 或兼容 Word 的线性格式,避免 LaTeX 专属语法).
"""

In [ ]:
import os
import re
import httpx
import subprocess
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

print("🚀 启动 Step 5.5：Qwen-Long 润色并直出完美 Word 报告...")

# =====================================================================
# 1. 🚨 核心修复：重新初始化 qwen_client
# =====================================================================
load_dotenv(find_dotenv(), override=True)
qwen_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    http_client=httpx.Client(timeout=1800.0)
)

# 假设 step5_qwen_prompt 在你前面的 Cell 已经定义过了
messages = [{"role": "user", "content": step5_qwen_prompt}]

print("⏳ Qwen-Long 正在奋笔疾书中 (大概需要几十秒)...")
response = qwen_client.chat.completions.create(
    model="qwen-long",
    messages=messages,
    temperature=0.3,
)

polished_report_text = response.choices[0].message.content

# =====================================================================
# 2. 🌟 核心防爆修复：在内存中直接洗白公式边界符号，然后再存！
# =====================================================================
# 将 \[ 和 \] 替换为 $$
polished_report_text = re.sub(r'\\\[', '$$$$', polished_report_text)
polished_report_text = re.sub(r'\\\]', '$$$$', polished_report_text)
# 将 \( 和 \) 替换为 $ （行内公式修复）
polished_report_text = re.sub(r'\\\(', '$', polished_report_text)
polished_report_text = re.sub(r'\\\)', '$', polished_report_text)

# 3. 落盘为“洗白后”的 Markdown
final_md_filename = "Final_Planning_Report_Polished.md"
with open(final_md_filename, "w", encoding="utf-8") as f:
    f.write(polished_report_text)

# =====================================================================
# 4. 无缝衔接：立刻调用 Pandoc 直出 Word
# =====================================================================
word_filename = "Final_Planning_Report_Perfect.docx"
print("🛠️ 正在将修复好的 Markdown 转换为 Word 文档...")

try:
    subprocess.run(["pandoc", final_md_filename, "-o", word_filename], check=True)
    print("\n" + "═"*60)
    print(f"🎉 伟大胜利！带完美排版和原生公式的 Word 文档已生成！")
    print("═"*60 + "\n")
    print(f"📁 最终成果：{word_filename}")
except Exception as e:
    print(f"\n❌ Word 转换失败: {e}")